# Embedding Search Evaluation

## utils.py import

평가할 함수인 search_embedding import

In [30]:
from utils import search_embedding

In [31]:
# print(search_embedding("category", "피자 파는 곳", 3))

## 평가셋 import

In [32]:
import pandas as pd

In [33]:
# 정답 라벨 파서
def parse_label(instr:str) -> tuple[tuple, tuple]:
    if instr == "":
        return tuple([]), tuple([])

    labels = instr.split(";")[:-1]

    codes = []
    weight = []
    for l in labels:
        lsplit = l.split(":")
        codes.append(lsplit[0])
        weight.append(lsplit[1])

    return tuple(codes), tuple(weight)

In [34]:
category_eval_df = pd.read_csv("eval_csv/category_eval.csv", index_col="index")

In [35]:
print(category_eval_df["label"].tolist())

['CAT0000:5;CAT0034:5;CAT0073:3;', 'CAT0001:5;CAT0085:4;CAT0047:2;', 'CAT0002:5;CAT0119:4;CAT0052:3;', 'CAT0003:5;CAT0040:4;CAT0012:3;', 'CAT0005:5;CAT0042:5;CAT0028:3;', 'CAT0007:5;CAT0015:4;CAT0113:3;', 'CAT0008:5;CAT0093:4;CAT0097:3;', 'CAT0009:5;CAT0046:4;CAT0114:2;', 'CAT0011:5;CAT0022:4;CAT0084:2;', 'CAT0012:5;CAT0109:4;CAT0040:3;', 'CAT0019:5;CAT0063:4;CAT0100:4;', 'CAT0020:5;CAT0067:5;', 'CAT0024:5;CAT0083:4;CAT0047:3;', 'CAT0031:5;CAT0043:4;CAT0074:4;', 'CAT0048:5;CAT0059:4;CAT0100:2;', 'CAT0035:5;CAT0027:4;CAT0049:3;', 'CAT0052:5;CAT0070:4;CAT0095:4;', 'CAT0059:5;CAT0048:4;CAT0100:2;', 'CAT0061:5;CAT0089:4;CAT0046:2;', 'CAT0087:5;CAT0098:4;CAT0050:2;', 'CAT0095:5;CAT0052:4;CAT0070:3;', 'CAT0104:5;CAT0017:4;CAT0082:2;', 'CAT0114:5;CAT0105:4;CAT0046:4;', 'CAT0090:5;CAT0088:4;CAT0065:2;', 'CAT0110:5;CAT0039:3;CAT0108:2;', 'CAT0121:5;CAT0078:4;CAT0091:2;', 'CAT0122:5;CAT0027:2;CAT0005:2;', 'CAT0083:5;CAT0024:4;CAT0047:2;']


In [36]:
eval_keys = ["precision@k", "recall@k", "MRR", "average_precision"]
def eval_tuple(prediction: list[str], label: list[str]) -> dict:
    rdict = {}

    pred_k = prediction
    label_set = set(label)
    top_n = len(pred_k)
    label_cnt = len(label_set)

    # 중복 제거 (prediction 기준)
    seen = set()
    hits = []
    for p in pred_k:
        if p in seen:
            continue
        seen.add(p)
        if p in label_set:
            hits.append(p)

    hit_cnt = len(hits)

    # Precision@k
    rdict["precision@k"] = round(hit_cnt / top_n, 4) if top_n > 0 else 0

    # Recall@k
    rdict["recall@k"] = round(hit_cnt / label_cnt, 4) if label_cnt > 0 else None

    # MRR
    mrr = 0
    for i, p in enumerate(pred_k):
        if p in label_set:
            mrr = 1 / (i + 1)
            break
    rdict["MRR"] = round(mrr, 4)

    # Average Precision
    ap_sum = 0
    hit_count = 0
    seen = set()

    for i, p in enumerate(pred_k, start=1):
        if p in seen:
            continue
        seen.add(p)

        if p in label_set:
            hit_count += 1
            ap_sum += hit_count / i

    rdict["average_precision"] = (
        round(ap_sum / label_cnt, 4) if label_cnt > 0 else None
    )

    return rdict

In [ ]:
query = category_eval_df["query"].tolist()[0]
print(query)

labels = category_eval_df["label"].tolist()[0]
print(labels)

prediction = ['CAT0034', 'CAT0076', 'CAT0000'] # search_embedding("category", query, 3)
print(prediction)

회식으로 시끌벅적하게 고기 구워 먹을 만한 데 찾고 있어
CAT0000:5;CAT0034:5;CAT0073:3;


['CAT0034', 'CAT0076', 'CAT0000']


In [38]:
label_tuples = parse_label(labels)
print(label_tuples)

(('CAT0000', 'CAT0034', 'CAT0073'), ('5', '5', '3'))


In [39]:
print(eval_tuple(prediction, label_tuples[0]))

{'precision@k': 0.6667, 'recall@k': 0.6667, 'MRR': 1.0, 'average_precision': 0.5556}


## tuple 순회 로직

In [50]:
def run_full_evaluation(table_name: str, top_n: int, queries: list[str], labels: list[str]):
    sumdict = {k: 0.0 for k in eval_keys}

    for i in range(len(queries)):
        prediction = search_embedding(table_name, queries[i], top_n)
        label_list = parse_label(labels[i])[0]

        buff = eval_tuple(prediction, label_list)

        metric_text = "\t".join([f"{k}: {buff[k]:.2f}" for k in eval_keys])
        print(f"{table_name} tuple{i:>5}: {metric_text}")

        for k in eval_keys:
            sumdict[k] += buff[k]

    for k in eval_keys:
        sumdict[k] = round(sumdict[k] / len(queries), 2)

    avg_text = "\t".join([f"{k}: {sumdict[k]}" for k in eval_keys])
    print(f"{table_name} {'average':<10}: {avg_text}")

    return sumdict

In [ ]:
# category_res = run_full_evaluation("category", 3, category_eval_df["query"].tolist(), category_eval_df["label"].tolist())

category tuple    0: precision@k: 0.67	recall@k: 0.67	MRR: 1.00	average_precision: 0.56
category tuple    1: precision@k: 0.33	recall@k: 0.33	MRR: 1.00	average_precision: 0.33
category tuple    2: precision@k: 0.33	recall@k: 0.33	MRR: 1.00	average_precision: 0.33
category tuple    3: precision@k: 0.33	recall@k: 0.33	MRR: 1.00	average_precision: 0.33
category tuple    4: precision@k: 0.00	recall@k: 0.00	MRR: 0.00	average_precision: 0.00
category tuple    5: precision@k: 0.67	recall@k: 0.67	MRR: 1.00	average_precision: 0.67
category tuple    6: precision@k: 0.33	recall@k: 0.33	MRR: 1.00	average_precision: 0.33
category tuple    7: precision@k: 0.33	recall@k: 0.33	MRR: 1.00	average_precision: 0.33
category tuple    8: precision@k: 0.00	recall@k: 0.00	MRR: 0.00	average_precision: 0.00
category tuple    9: precision@k: 0.67	recall@k: 0.67	MRR: 1.00	average_precision: 0.67
category tuple   10: precision@k: 0.67	recall@k: 0.67	MRR: 1.00	average_precision: 0.67
category tuple   11: precision@k

## 각 table에 대해서 순서대로 평가

In [55]:
top_n = 3

In [56]:
menu_df = pd.read_csv("eval_csv/menu_eval.csv", index_col="index")
menu_res = run_full_evaluation("menu", top_n, menu_df["query"].tolist(), menu_df["label"].tolist())

menu tuple    0: precision@k: 0.00	recall@k: 0.00	MRR: 0.00	average_precision: 0.00
menu tuple    1: precision@k: 0.00	recall@k: 0.00	MRR: 0.00	average_precision: 0.00
menu tuple    2: precision@k: 0.00	recall@k: 0.00	MRR: 0.00	average_precision: 0.00
menu tuple    3: precision@k: 0.67	recall@k: 0.67	MRR: 0.50	average_precision: 0.39
menu tuple    4: precision@k: 0.67	recall@k: 0.67	MRR: 1.00	average_precision: 0.67
menu tuple    5: precision@k: 0.33	recall@k: 0.33	MRR: 0.50	average_precision: 0.17
menu tuple    6: precision@k: 0.00	recall@k: 0.00	MRR: 0.00	average_precision: 0.00
menu tuple    7: precision@k: 0.00	recall@k: 0.00	MRR: 0.00	average_precision: 0.00
menu tuple    8: precision@k: 0.33	recall@k: 0.33	MRR: 0.50	average_precision: 0.17
menu tuple    9: precision@k: 0.00	recall@k: 0.00	MRR: 0.00	average_precision: 0.00
menu tuple   10: precision@k: 0.67	recall@k: 0.67	MRR: 0.50	average_precision: 0.39
menu tuple   11: precision@k: 0.00	recall@k: 0.00	MRR: 0.00	average_precisio

In [57]:
food_df = pd.read_csv("eval_csv/food_eval.csv", index_col="index")
food_res = run_full_evaluation("food", top_n, food_df["query"].tolist(), food_df["label"].tolist())

food tuple    0: precision@k: 0.00	recall@k: 0.00	MRR: 0.00	average_precision: 0.00
food tuple    1: precision@k: 0.67	recall@k: 0.67	MRR: 0.50	average_precision: 0.39
food tuple    2: precision@k: 0.33	recall@k: 0.33	MRR: 0.33	average_precision: 0.11
food tuple    3: precision@k: 0.67	recall@k: 0.67	MRR: 0.50	average_precision: 0.39
food tuple    4: precision@k: 0.67	recall@k: 0.67	MRR: 1.00	average_precision: 0.67
food tuple    5: precision@k: 0.33	recall@k: 0.33	MRR: 1.00	average_precision: 0.33
food tuple    6: precision@k: 1.00	recall@k: 1.00	MRR: 1.00	average_precision: 1.00
food tuple    7: precision@k: 0.00	recall@k: 0.00	MRR: 0.00	average_precision: 0.00
food tuple    8: precision@k: 0.33	recall@k: 0.33	MRR: 1.00	average_precision: 0.33
food tuple    9: precision@k: 0.00	recall@k: 0.00	MRR: 0.00	average_precision: 0.00
food tuple   10: precision@k: 0.67	recall@k: 0.67	MRR: 0.50	average_precision: 0.39
food tuple   11: precision@k: 0.00	recall@k: 0.00	MRR: 0.00	average_precisio

In [58]:
category_df = pd.read_csv("eval_csv/category_eval.csv", index_col="index")
category_res = run_full_evaluation("category", top_n, category_df["query"].tolist(), category_df["label"].tolist())

category tuple    0: precision@k: 0.67	recall@k: 0.67	MRR: 1.00	average_precision: 0.56
category tuple    1: precision@k: 0.33	recall@k: 0.33	MRR: 1.00	average_precision: 0.33
category tuple    2: precision@k: 0.33	recall@k: 0.33	MRR: 1.00	average_precision: 0.33
category tuple    3: precision@k: 0.33	recall@k: 0.33	MRR: 1.00	average_precision: 0.33
category tuple    4: precision@k: 0.00	recall@k: 0.00	MRR: 0.00	average_precision: 0.00
category tuple    5: precision@k: 0.67	recall@k: 0.67	MRR: 1.00	average_precision: 0.67
category tuple    6: precision@k: 0.33	recall@k: 0.33	MRR: 1.00	average_precision: 0.33
category tuple    7: precision@k: 0.33	recall@k: 0.33	MRR: 1.00	average_precision: 0.33
category tuple    8: precision@k: 0.00	recall@k: 0.00	MRR: 0.00	average_precision: 0.00
category tuple    9: precision@k: 0.67	recall@k: 0.67	MRR: 1.00	average_precision: 0.67
category tuple   10: precision@k: 0.67	recall@k: 0.67	MRR: 1.00	average_precision: 0.67
category tuple   11: precision@k

In [59]:
tag_df = pd.read_csv("eval_csv/tag_eval.csv", index_col="index")
tag_res = run_full_evaluation("tag", top_n, tag_df["query"].tolist(), tag_df["label"].tolist())

tag tuple    0: precision@k: 0.33	recall@k: 0.50	MRR: 0.50	average_precision: 0.25
tag tuple    1: precision@k: 0.33	recall@k: 0.33	MRR: 0.33	average_precision: 0.11
tag tuple    2: precision@k: 0.33	recall@k: 0.33	MRR: 1.00	average_precision: 0.33
tag tuple    3: precision@k: 0.33	recall@k: 0.33	MRR: 1.00	average_precision: 0.33
tag tuple    4: precision@k: 0.00	recall@k: 0.00	MRR: 0.00	average_precision: 0.00
tag tuple    5: precision@k: 0.67	recall@k: 0.67	MRR: 0.50	average_precision: 0.39
tag tuple    6: precision@k: 0.33	recall@k: 0.33	MRR: 1.00	average_precision: 0.33
tag tuple    7: precision@k: 0.33	recall@k: 0.33	MRR: 1.00	average_precision: 0.33
tag tuple    8: precision@k: 0.67	recall@k: 0.67	MRR: 1.00	average_precision: 0.56
tag tuple    9: precision@k: 0.33	recall@k: 0.33	MRR: 1.00	average_precision: 0.33
tag tuple   10: precision@k: 0.00	recall@k: 0.00	MRR: 0.00	average_precision: 0.00
tag tuple   11: precision@k: 0.33	recall@k: 0.33	MRR: 0.50	average_precision: 0.17
tag 

In [60]:
review_df = pd.read_csv("eval_csv/review_eval.csv", index_col="index")
review_res = run_full_evaluation("review", top_n, review_df["query"].tolist(), review_df["label"].tolist())

review tuple    0: precision@k: 1.00	recall@k: 1.00	MRR: 1.00	average_precision: 1.00
review tuple    1: precision@k: 0.67	recall@k: 0.67	MRR: 1.00	average_precision: 0.67
review tuple    2: precision@k: 1.00	recall@k: 1.00	MRR: 1.00	average_precision: 1.00
review tuple    3: precision@k: 0.33	recall@k: 0.33	MRR: 1.00	average_precision: 0.33
review tuple    4: precision@k: 0.00	recall@k: 0.00	MRR: 0.00	average_precision: 0.00
review tuple    5: precision@k: 1.00	recall@k: 1.00	MRR: 1.00	average_precision: 1.00
review tuple    6: precision@k: 0.67	recall@k: 0.67	MRR: 0.50	average_precision: 0.39
review tuple    7: precision@k: 0.33	recall@k: 0.33	MRR: 1.00	average_precision: 0.33
review tuple    8: precision@k: 0.33	recall@k: 0.33	MRR: 1.00	average_precision: 0.33
review tuple    9: precision@k: 0.00	recall@k: 0.00	MRR: 0.00	average_precision: 0.00
review tuple   10: precision@k: 0.67	recall@k: 0.67	MRR: 1.00	average_precision: 0.67
review tuple   11: precision@k: 0.67	recall@k: 0.67	MR